# Reproducing PerLeadCNN Results

This notebook reproduces the exact evaluation metrics for the best PerLeadCNN model
from the 30-split patient-grouped validation (commit `dbb6f49`).

**What it verifies:**
- Data loading and preprocessing pipeline
- Test split reconstruction from saved seeds
- Model inference and metric computation
- All values match `per_split.json` and `summary.json`


In [1]:
import json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.signal import butter, iirnotch, sosfiltfilt, tf2sos, resample
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, confusion_matrix)

# --------------- data loading (inlined from prepare.py) ---------------
N_LEADS = 12
SEQ_LEN = 5000
FS = 500.0
SD_LEAD_ORDER = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]
SD_LABEL_POS = "Preeclampsia or Other Hypertensive Disorders of Pregnancy"

def _apply_sos(sos, X):
    N, C, T = X.shape
    flat = X.reshape(N * C, T)
    X[:] = sosfiltfilt(sos, flat, axis=-1).reshape(N, C, T)
    return X

def preprocess(X):
    X = X.copy()
    sos_hp = butter(4, 0.5, btype="high", fs=FS, output="sos")
    X = _apply_sos(sos_hp, X)
    b, a = iirnotch(60.0, 30.0, fs=FS)
    sos_notch = tf2sos(b, a)
    X = _apply_sos(sos_notch, X)
    mean = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2, keepdims=True) + 1e-8
    X = (X - mean) / std
    return X

def load_ecg_data(data_dir):
    data_dir = os.path.normpath(data_dir)
    ekg_dir = os.path.join(data_dir, "ekg_data")
    meta_path = os.path.join(data_dir, "metadata.csv")
    if not os.path.exists(meta_path):
        meta_path = os.path.join(data_dir, "metadata_balanced.csv")
    meta = pd.read_csv(meta_path)
    available = {
        int(os.path.splitext(f)[0])
        for f in os.listdir(ekg_dir) if f.endswith(".csv")
    }
    meta = meta[meta["ECGTestID"].apply(lambda x: int(x) in available)].copy()
    X_list, y_list, pat_list, upsampled_list = [], [], [], []
    for _, row in meta.iterrows():
        path = os.path.join(ekg_dir, f"{int(row['ECGTestID'])}.csv")
        try:
            df = pd.read_csv(path, skipinitialspace=True, usecols=SD_LEAD_ORDER)
            arr = df[SD_LEAD_ORDER].values.T.astype(np.float32)
            if arr.shape[0] != 12:
                continue
            n_timepoints = arr.shape[1]
            if n_timepoints == SEQ_LEN:
                was_upsampled = False
            elif n_timepoints == 2500:
                arr = resample(arr, SEQ_LEN, axis=1).astype(np.float32)
                was_upsampled = True
            else:
                continue
            if arr.shape != (12, SEQ_LEN):
                continue
            X_list.append(arr)
            y_list.append(1 if row["PatLabel"] == SD_LABEL_POS else 0)
            pat_list.append(row["Pat_Obfus_MRN"])
            upsampled_list.append(was_upsampled)
        except Exception:
            continue
    X = np.stack(X_list)
    y = np.array(y_list, dtype=np.int64)
    patient_ids = np.array(pat_list)
    is_upsampled = np.array(upsampled_list, dtype=bool)
    mask = np.isfinite(X).all(axis=2).all(axis=1)
    X, y, patient_ids, is_upsampled = X[mask], y[mask], patient_ids[mask], is_upsampled[mask]
    flat_mask = (X.std(axis=2) < 1e-4).any(axis=1)
    keep = ~flat_mask
    X, y, patient_ids, is_upsampled = X[keep], y[keep], patient_ids[keep], is_upsampled[keep]
    try:
        nan_mask = np.isnan(patient_ids.astype(float))
    except (ValueError, TypeError):
        nan_mask = np.array([str(p).strip() in ("", "nan", "None") for p in patient_ids])
    keep = ~nan_mask
    X, y, patient_ids, is_upsampled = X[keep], y[keep], patient_ids[keep], is_upsampled[keep]
    return X, y, patient_ids, is_upsampled

print("All functions defined (self-contained, no external imports needed)")

All functions defined (self-contained, no external imports needed)


## 1. Load Data


In [2]:
DATA_DIR = "data/seniordesign_upload"
X_all, y_all, patient_ids, _ = load_ecg_data(DATA_DIR)
X_all = preprocess(X_all)
X_all = X_all[:, :, ::2]  # downsample 500 Hz -> 250 Hz

print(f"Samples: {len(y_all)}")
print(f"Positive: {int(y_all.sum())} ({y_all.mean():.1%})")
print(f"Negative: {int((y_all==0).sum())}")
print(f"Unique patients: {len(np.unique(patient_ids))}")
print(f"Shape: {X_all.shape}  (samples, 12 leads, 2500 timepoints @ 250 Hz)")

Samples: 2178
Positive: 335 (15.4%)
Negative: 1843
Unique patients: 1383
Shape: (2178, 12, 2500)  (samples, 12 leads, 2500 timepoints @ 250 Hz)


## 2. Define Model


In [3]:
class PerLeadCNN(nn.Module):
    def __init__(self, n_leads=12, filters=(16, 32, 48), kernels=(31, 21, 11),
                 dropout=0.15, n_classes=2):
        super().__init__()
        layers = []
        in_ch = 1
        for f, k in zip(filters, kernels):
            layers.extend([
                nn.Conv1d(in_ch, f, k, stride=2, padding=k // 2, bias=False),
                nn.BatchNorm1d(f), nn.Mish(),
            ])
            in_ch = f
        self.backbone = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.n_leads = n_leads
        self.head_drop = nn.Dropout(dropout)
        self.fc = nn.Linear(filters[-1] * n_leads, n_classes)

    def forward(self, x):
        B, L, T = x.shape
        x = x.reshape(B * L, 1, T)
        x = self.backbone(x)
        x = self.pool(x).squeeze(-1)
        x = x.reshape(B, L * x.shape[-1])
        return self.fc(self.head_drop(x))

print(f"Parameters: {sum(p.numel() for p in PerLeadCNN().parameters()):,}")

Parameters: 29,490


## 3. Load Saved Results for Comparison


In [4]:
RESULTS_DIR = "multisplit_dbb6f49"

with open(f"{RESULTS_DIR}/summary.json") as f:
    summary = json.load(f)
with open(f"{RESULTS_DIR}/per_split.json") as f:
    per_split = json.load(f)

print(f"Recorded: AUROC {summary['auroc_mean']:.4f} +/- {summary['auroc_std']:.4f}")
print(f"Recorded: AUPRC {summary['auprc_mean']:.4f} +/- {summary['auprc_std']:.4f}")
print(f"Splits: {summary['n_splits']}")

Recorded: AUROC 0.7085 +/- 0.0493
Recorded: AUPRC 0.3423 +/- 0.0751
Splits: 30


## 4. Reproduce Split Helper

For each split, reconstruct the exact test set using the saved seed,
run inference with the saved model weights, and compare metrics.


In [5]:
def evaluate_on_split(split_i, model_path):
    split_seed = split_i * 7 + 1000

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=split_seed)
    dev_idx, test_idx = next(iter(sgkf.split(X_all, y_all, groups=patient_ids)))
    X_te, y_te = X_all[test_idx], y_all[test_idx]

    model = PerLeadCNN(filters=(16, 32, 48), kernels=(31, 21, 11), dropout=0.15)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()

    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(X_te, dtype=torch.float32)), dim=1)[:, 1].numpy()

    auroc = roc_auc_score(y_te, probs)
    auprc = average_precision_score(y_te, probs)
    fpr, tpr, thresholds = roc_curve(y_te, probs)

    # Youden threshold
    j_idx = np.argmax(tpr - fpr)
    thr_youden = thresholds[j_idx]

    # Sens >= 0.80 threshold
    valid = tpr >= 0.80
    if valid.any():
        candidates = np.where(valid)[0]
        thr_sens80 = thresholds[candidates[np.argmax(1 - fpr[candidates])]]
    else:
        thr_sens80 = thr_youden

    results = {"auroc": auroc, "auprc": auprc}
    for name, thr in [("youden", thr_youden), ("sens80", thr_sens80)]:
        preds = (probs >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, preds).ravel()
        sens = tp / (tp + fn)
        spec = tn / (tn + fp)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        f1 = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0
        results.update({
            f"{name}_sens": sens, f"{name}_spec": spec,
            f"{name}_prec": prec, f"{name}_npv": npv,
            f"{name}_f1": f1, f"{name}_threshold": float(thr),
        })
    return results, len(y_te), int(y_te.sum())

print("evaluate_on_split() defined")

evaluate_on_split() defined


## 5. Verify Best Model (Split 17, AUROC 0.7793)


In [6]:
best_results, n_test, n_pos = evaluate_on_split(17, f"{RESULTS_DIR}/best_model.pt")
expected = per_split[17]

print(f"Test set: {n_test} samples ({n_pos} positive)")
print(f"Seed: {17 * 7 + 1000}")
print()
print(f"{'Metric':<25} {'Reproduced':>12} {'Recorded':>12} {'Match':>8}")
print("-" * 60)
for key in ["auroc", "auprc", "youden_sens", "youden_spec", "youden_prec",
            "youden_npv", "youden_f1", "sens80_sens", "sens80_spec",
            "sens80_prec", "sens80_npv", "sens80_f1"]:
    repro = best_results[key]
    rec = expected[key]
    match = "OK" if abs(repro - rec) < 1e-4 else "MISMATCH"
    print(f"{key:<25} {repro:12.4f} {rec:12.4f} {match:>8}")

Test set: 446 samples (70 positive)
Seed: 1119

Metric                      Reproduced     Recorded    Match
------------------------------------------------------------
auroc                           0.7793       0.7793       OK
auprc                           0.4852       0.4852       OK
youden_sens                     0.5429       0.5429       OK
youden_spec                     0.8963       0.8963       OK
youden_prec                     0.4935       0.4935       OK
youden_npv                      0.9133       0.9133       OK
youden_f1                       0.5170       0.5170       OK
sens80_sens                     0.8000       0.8000       OK
sens80_spec                     0.5559       0.5559       OK
sens80_prec                     0.2511       0.2511       OK
sens80_npv                      0.9372       0.9372       OK
sens80_f1                       0.3823       0.3823       OK


## 6. Verify Median Model (Split 24, AUROC 0.7251)


In [7]:
median_results, n_test, n_pos = evaluate_on_split(24, f"{RESULTS_DIR}/median_model.pt")
expected_med = per_split[24]

print(f"Test set: {n_test} samples ({n_pos} positive)")
print(f"Seed: {24 * 7 + 1000}")
print()
print(f"{'Metric':<25} {'Reproduced':>12} {'Recorded':>12} {'Match':>8}")
print("-" * 60)
for key in ["auroc", "auprc", "youden_sens", "youden_spec", "youden_prec",
            "youden_npv", "youden_f1", "sens80_sens", "sens80_spec",
            "sens80_prec", "sens80_npv", "sens80_f1"]:
    repro = median_results[key]
    rec = expected_med[key]
    match = "OK" if abs(repro - rec) < 1e-4 else "MISMATCH"
    print(f"{key:<25} {repro:12.4f} {rec:12.4f} {match:>8}")

Test set: 464 samples (78 positive)
Seed: 1168

Metric                      Reproduced     Recorded    Match
------------------------------------------------------------
auroc                           0.7251       0.7251       OK
auprc                           0.2965       0.2965       OK
youden_sens                     0.8205       0.8205       OK
youden_spec                     0.5466       0.5466       OK
youden_prec                     0.2678       0.2678       OK
youden_npv                      0.9378       0.9378       OK
youden_f1                       0.4038       0.4038       OK
sens80_sens                     0.8077       0.8077       OK
sens80_spec                     0.5492       0.5492       OK
sens80_prec                     0.2658       0.2658       OK
sens80_npv                      0.9339       0.9339       OK
sens80_f1                       0.4000       0.4000       OK


## 7. Verify Aggregate Metrics

Recompute mean/std from per_split.json and confirm they match summary.json.


In [8]:
recorded_aurocs = [s["auroc"] for s in per_split]
recorded_auprcs = [s["auprc"] for s in per_split]

checks = [
    ("AUROC mean", np.mean(recorded_aurocs), summary["auroc_mean"]),
    ("AUROC std",  np.std(recorded_aurocs),  summary["auroc_std"]),
    ("AUPRC mean", np.mean(recorded_auprcs), summary["auprc_mean"]),
    ("AUPRC std",  np.std(recorded_auprcs),  summary["auprc_std"]),
]

print(f"{'Metric':<15} {'Computed':>12} {'Recorded':>12} {'Match':>8}")
print("-" * 50)
for name, computed, recorded in checks:
    match = "OK" if abs(computed - recorded) < 1e-6 else "MISMATCH"
    print(f"{name:<15} {computed:12.6f} {recorded:12.6f} {match:>8}")

Metric              Computed     Recorded    Match
--------------------------------------------------
AUROC mean          0.708514     0.708514       OK
AUROC std           0.049281     0.049281       OK
AUPRC mean          0.342257     0.342257       OK
AUPRC std           0.075088     0.075088       OK


## 8. Reproduction Recipe

To reproduce any split's test set:
```python
split_seed = split_i * 7 + 1000
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=split_seed)
dev_idx, test_idx = next(iter(sgkf.split(X_all, y_all, groups=patient_ids)))
```

To reproduce model training for a split:
```python
torch.manual_seed(split_seed + 2)
np.random.seed(split_seed + 2)
torch.cuda.manual_seed_all(split_seed + 2)
```

| Component | Value |
|---|---|
| Outer CV | StratifiedGroupKFold, 5 folds, first fold = test |
| Inner CV | StratifiedGroupKFold, 8 folds, first fold = val |
| Seed formula | `split_i * 7 + 1000` |
| Best model | Split 17 (seed 1119), AUROC 0.7793 |
| Median model | Split 24 (seed 1168), AUROC 0.7251 |
